# 00 · Start projektu: obszar i szybki test danych

**Sztuczna inteligencja w przetwarzaniu geodanych** · projekt zespołowy · P1 · ok. 20 minut

Sprawdzicie, czy główne dane do Waszego tematu da się pobrać i pokazać na mapie obszaru projektu. Na końcu notatnik wypisze **metryczkę danych** do wklejenia w kartę projektu i zapisze mapę.

**Jak pracować**
1. *Plik → Zapisz kopię na Dysku* i pracujcie na kopii.
2. Uruchamiajcie komórki po kolei: kliknięcie w komórkę i **Shift+Enter**.
3. Zmieniacie tylko wiersze oznaczone `# USTAW`.
4. Gdy serwer danych nie odpowiada, notatnik napisze, co zrobić; zwykle wystarczy spróbować ponownie za minutę.

## 1. Instalacja i ustawienia

In [ ]:
%pip install -q osmnx mapclassify

In [1]:
import io
import re
import tempfile
import time
import zipfile
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import osmnx as ox
import pandas as pd
import requests
from IPython.display import Markdown, display
from shapely.geometry import box

ZIARNO = 42                 # ten sam wynik losowania przy każdym uruchomieniu
UKLAD = "EPSG:2180"         # PL-1992: współrzędne w metrach
np.random.seed(ZIARNO)

ROOT = Path.cwd()

if ROOT.name == "notebooks":
    ROOT = ROOT.parent

FOLDER = ROOT / "wyniki"
FOLDER.mkdir(parents=True, exist_ok=True)

print("Wyniki trafią do:", FOLDER)

# Serwery Overpass API, z których OSMnx pobiera dane OSM (główny i dwa zapasowe), jak na L1 i L2
SERWERY_OVERPASS = ["https://overpass-api.de/api", "https://overpass.private.coffee/api",
                    "https://maps.mail.ru/osm/tools/overpass/api"]
ox.settings.use_cache = True                             # pobrane dane zostają na Dysku
ox.settings.cache_folder = str(FOLDER / "cache_osm")
ox.settings.requests_timeout = 300
ox.settings.log_console = False


def z_osm(funkcja, *args, **kwargs):
    """Wywołuje funkcję OSMnx; gdy serwer nie odpowiada, próbuje kolejnego."""
    for i, url in enumerate(SERWERY_OVERPASS):
        ox.settings.overpass_url = url
        ox.settings.overpass_rate_limit = (i == 0)       # tylko serwer główny ma limit zapytań
        try:
            return funkcja(*args, **kwargs)
        except ox._errors.InsufficientResponseError:      # serwer odpowiedział, ale takich obiektów nie ma
            raise ValueError("w obszarze nie ma obiektów OSM z podanymi tagami; sprawdźcie OSM_TAGI") from None
        except Exception as blad:
            dalej = " Próbuję następnego…" if i < len(SERWERY_OVERPASS) - 1 else ""
            print(f"Serwer {url} nie odpowiedział ({type(blad).__name__}).{dalej}")
            time.sleep(3)
    raise ConnectionError("Żaden serwer OSM nie odpowiedział.")

Wyniki trafią do: e:\Semestr7\Locker-Net\wyniki


## 2. Obszar projektu

Domyślnie Kraków, ten sam obszar co na laboratoriach. Inną nazwę wpiszcie tylko wtedy, gdy temat tego wymaga, np. `"województwo małopolskie, Polska"`. Jeśli serwer OpenStreetMap nie odpowie, notatnik użyje prostokąta wokół Krakowa i napisze to.

In [2]:
NAZWA_OBSZARU = "Kraków, Polska"                    # USTAW, jeśli temat wymaga innego obszaru
RAMKA_ZAPASOWA = (19.79, 49.97, 20.22, 50.13)       # dł. min, szer. min, dł. max, szer. max (Kraków)

try:
    obszar = ox.geocode_to_gdf(NAZWA_OBSZARU)[["geometry"]]
    obszar["nazwa"] = NAZWA_OBSZARU
except Exception as blad:
    print(f"Serwer OSM nie zwrócił granicy ({type(blad).__name__}). Używam prostokąta zapasowego.")
    obszar = gpd.GeoDataFrame({"nazwa": [NAZWA_OBSZARU + " (prostokąt zapasowy)"]},
                              geometry=[box(*RAMKA_ZAPASOWA)], crs="EPSG:4326")

obszar_m = obszar.to_crs(UKLAD)
print(f"Obszar: {obszar['nazwa'].iloc[0]}; powierzchnia {obszar_m.area.iloc[0] / 1e6:.0f} km²")

Obszar: Kraków, Polska; powierzchnia 326 km²


## 3. Wasze dane

Wybierzcie **jedno** główne źródło, najlepiej to, z którego weźmiecie zmienną celu:

| `ZRODLO` | Co wczyta notatnik |
| --- | --- |
| `"osm"` | obiekty z OpenStreetMap w obszarze projektu, według `OSM_TAGI`, np. przejścia dla pieszych albo parkingi |
| adres `https://…` albo ścieżka do pliku | GeoPackage, GeoJSON, SHP (także w ZIP), rozkład GTFS w ZIP (przystanki) albo CSV ze współrzędnymi |

Plik z Dysku Google: najpierw wgrajcie go do folderu `GeoAI_projekt`, a potem wpiszcie ścieżkę, np. `"/content/drive/MyDrive/GeoAI_projekt/wypadki.csv"`. Dane po rejestracji, np. eksport z Obserwatorium BRD, pobieracie ręcznie i wgrywacie tak samo.

Przykłady źródeł z laboratoriów:
- rozkład tramwajów ZTP Kraków: `"https://gtfs.ztp.krakow.pl/GTFS_KRK_T.zip"`,
- Generalny Pomiar Ruchu 2025, odcinki dróg krajowych (SHP w ZIP): `"https://www.gov.pl/attachment/7f8ca805-e384-4802-bf10-d36538e6ef84"`.

In [3]:
ZRODLO = "osm"
OSM_TAGI = {"amenity": "parcel_locker"}

KOL_X, KOL_Y = "lon", "lat"
UKLAD_CSV = "EPSG:4326"
SEPARATOR = None
PRZECINEK_DZIESIETNY = False
KODOWANIE = "utf-8"

KOL_DATA = None

In [4]:
def pobierz(zrodlo):
    """Plik z internetu zapisuje w folderze tymczasowym i zwraca ścieżkę; plik lokalny zwraca bez zmian."""
    if str(zrodlo).startswith(("http://", "https://")):
        odp = requests.get(zrodlo, timeout=180)
        odp.raise_for_status()
        sciezka = Path(tempfile.mkdtemp()) / (Path(zrodlo.split("?")[0]).name or "dane")
        sciezka.write_bytes(odp.content)
        print(f"Pobrano {len(odp.content) / 1e6:.1f} MB")
        return sciezka
    return Path(zrodlo)


def wczytaj_dane(zrodlo):
    if zrodlo == "osm":
        dane = z_osm(ox.features_from_polygon, obszar.geometry.iloc[0], tags=OSM_TAGI).reset_index()
        return dane[[k for k in dane.columns if k != "nodes"]]
    sciezka = pobierz(zrodlo)
    if zipfile.is_zipfile(sciezka):
        with zipfile.ZipFile(sciezka) as z:
            if "stops.txt" in z.namelist():          # rozkład GTFS: bierzemy przystanki
                przystanki = pd.read_csv(z.open("stops.txt"))
                return gpd.GeoDataFrame(przystanki, crs="EPSG:4326",
                                        geometry=gpd.points_from_xy(przystanki["stop_lon"], przystanki["stop_lat"]))
        folder = Path(tempfile.mkdtemp())                  # warstwa SHP, GPKG lub GeoJSON w ZIP, także w podfolderze
        with zipfile.ZipFile(sciezka) as z:
            z.extractall(folder)
        warstwy = sorted(w for r in ("*.shp", "*.gpkg", "*.geojson") for w in folder.rglob(r))
        if not warstwy:
            raise ValueError("W archiwum ZIP nie ma pliku .shp, .gpkg ani .geojson")
        print("Warstwy w archiwum:", [w.name for w in warstwy], "→ wczytuję pierwszą")
        sciezka = warstwy[0]
    if sciezka.suffix.lower() == ".shp":
        dane = gpd.read_file(sciezka)
        tekstowe = [k for k in dane.columns if k != "geometry" and pd.api.types.is_string_dtype(dane[k])]
        tekst = " ".join(dane[tekstowe].astype(str).head(200).stack().tolist())
        if re.search("[³¹¿ê£¥¯æñÊÆÑ\x8c\x8f\x9c\x9f]", tekst):   # polskie litery z Windows-1250 odczytane źle
            print("Poprawiam kodowanie polskich znaków (Windows-1250).")
            dane = gpd.read_file(sciezka, encoding="cp1250")
        return dane
    if sciezka.suffix.lower() in (".csv", ".txt"):
        tabela = pd.read_csv(sciezka, sep=SEPARATOR, engine="python", encoding=KODOWANIE,
                             decimal="," if PRZECINEK_DZIESIETNY else ".")
        return gpd.GeoDataFrame(tabela, crs=UKLAD_CSV, geometry=gpd.points_from_xy(tabela[KOL_X], tabela[KOL_Y]))
    if sciezka.suffix.lower() in (".tif", ".tiff", ".jpg", ".png"):
        raise ValueError("To plik rastrowy: obejrzyjcie go w QGIS, a do ortofotomapy użyjcie sekcji 6.")
    return gpd.read_file(sciezka)


try:
    dane = wczytaj_dane(ZRODLO)
except (ConnectionError, requests.RequestException) as blad:
    print(f"Serwer z danymi nie odpowiada ({type(blad).__name__}).")
    print("Odczekajcie kilka minut i uruchomcie tę komórkę ponownie; pobrane już części zostają w pamięci podręcznej. "
          "Na sam test danych możecie też zmniejszyć obszar w sekcji 2, np. „Stare Miasto, Kraków, Polska”.")
    raise RuntimeError("Danych nie wczytano: serwer nie odpowiada.") from None
except Exception as blad:
    print(f"Nie udało się wczytać danych: {type(blad).__name__}: {blad}")
    print("Co sprawdzić: adres lub ścieżkę, nazwy kolumn KOL_X i KOL_Y, separator i kodowanie.")
    raise RuntimeError("Danych nie wczytano: zobaczcie komunikat wyżej.") from None

if dane.crs is None:
    print("Uwaga: plik nie ma układu współrzędnych. Przyjmuję EPSG:4326; jeśli to błąd, ustawcie go ręcznie.")
    dane = dane.set_crs("EPSG:4326")
print(f"Wczytano {len(dane)} rekordów, układ {dane.crs.to_string()}")
dane.head()

Serwer https://overpass-api.de/api nie odpowiedział (ResponseStatusCodeError). Próbuję następnego…
Serwer https://overpass.private.coffee/api nie odpowiedział (ReadTimeout). Próbuję następnego…
Wczytano 1525 rekordów, układ EPSG:4326


,element,id,geometry,amenity,brand,brand:wikidata,level,opening_hours,operator,operator:wikidata,...,monitoring:air_quality,recording:automated,recording:remote,building:floor,addr:unit,source:ref,direction,note,access,building
0,node,1502425596,POINT (19.99941 50.06453),parcel_locker,Paczkomat InPost,Q110970254,0,24/7,InPost,Q3182097,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,node,1910676577,POINT (19.91019 50.07502),parcel_locker,Orlen Paczka,Q110457879,NaN,24/7,Orlen,Q971649,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,node,2388234278,POINT (19.88418 50.08119),parcel_locker,DPD Pickup Station,Q114273730,NaN,24/7,DPD,Q541030,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,node,2481644200,POINT (19.97525 50.07134),parcel_locker,Allegro One Box,Q110738715,NaN,24/7,UPS,Q155026,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,node,2494254356,POINT (19.96171 50.06645),parcel_locker,Paczkomat InPost,Q110970254,NaN,24/7,InPost,Q3182097,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
DATA = ROOT / "data"
DATA.mkdir(parents=True, exist_ok=True)

dane.to_file(
    DATA / "automaty_paczkowe.geojson",
    driver="GeoJSON"
)

print("Zapisano dane OSM.")

Zapisano dane OSM.


## 4. Metryczka danych

Tę tabelę wklejacie do karty projektu: pole „Dane” i sekcja „Szybki test danych”. Zwróćcie uwagę na trzy rzeczy: jaka część rekordów leży w obszarze projektu, ile jest braków i jaki jest okres.

In [6]:
def metryczka(dane, kol_data=None):
    w_metrach = dane.to_crs(UKLAD)
    bez_geometrii = int((dane.geometry.isna() | dane.geometry.is_empty).sum())
    w_obszarze = w_metrach.intersects(obszar_m.geometry.iloc[0]).mean()
    braki = dane.drop(columns="geometry").isna().mean().mul(100).sort_values(ascending=False)
    braki = braki[braki > 0].head(6)
    wiersze = [
        ("Źródło", f"{ZRODLO} {OSM_TAGI}" if ZRODLO == "osm" else str(ZRODLO)),
        ("Liczba rekordów", f"{len(dane):,}".replace(",", " ")),
        ("Typy geometrii", ", ".join(f"{t}: {n}" for t, n in dane.geom_type.value_counts().items())),
        ("Rekordy bez geometrii", str(bez_geometrii)),
        ("Układ współrzędnych pliku", dane.crs.to_string()),
        ("W obszarze projektu", f"{w_obszarze:.0%}"),
        ("Liczba kolumn", str(dane.shape[1] - 1)),
        ("Kolumny z brakami [%]", ", ".join(f"{k}: {v:.0f}" for k, v in braki.items()) or "brak braków"),
    ]
    if kol_data:
        daty = pd.to_datetime(dane[kol_data], errors="coerce", dayfirst=True)
        wiersze.append(("Okres", f"{daty.min():%Y-%m-%d} – {daty.max():%Y-%m-%d}; bez daty: {daty.isna().mean():.0%}"))
    return "| Cecha | Wartość |\n| --- | --- |\n" + "\n".join(f"| {a} | {b} |" for a, b in wiersze)


tabela_metryczki = metryczka(dane, KOL_DATA)
display(Markdown(tabela_metryczki))

| Cecha | Wartość |
| --- | --- |
| Źródło | osm {'amenity': 'parcel_locker'} |
| Liczba rekordów | 1 525 |
| Typy geometrii | Point: 1435, Polygon: 90 |
| Rekordy bez geometrii | 0 |
| Układ współrzędnych pliku | EPSG:4326 |
| W obszarze projektu | 100% |
| Liczba kolumn | 82 |
| Kolumny z brakami [%] | access: 100, direction: 100, currency:PLN: 100, brand:wikipedia: 100, operator:type: 100, artwork_subject: 100 |

## 5. Mapa

Mapa interaktywna pokazuje najwyżej 5000 losowych rekordów. Zrzut statyczny trafia do pliku `00_start_mapa.png`; dołączcie go do karty projektu.

In [7]:

probka = dane[~(dane.geometry.isna() | dane.geometry.is_empty)]

if len(probka) > 5000:
    probka = probka.sample(5000, random_state=ZIARNO)

kolumny = [k for k in probka.columns if k != "geometry"][:6]

do_mapy = gpd.GeoDataFrame(
    probka[kolumny].astype(str),
    geometry=probka.geometry,
    crs=probka.crs
)

mapa = obszar.explore(
    tiles="Esri.WorldGrayCanvas",
    style_kwds={"fill": False, "color": "black", "weight": 2},
    name="Granice Krakowa"
)

do_mapy.explore(
    m=mapa,
    color="crimson",
    name="Automaty paczkowe"
)

fig, ax = plt.subplots(figsize=(8, 8))

obszar_m.boundary.plot(
    ax=ax,
    color="black",
    linewidth=1,
    label="Granice Krakowa"
)

probka.to_crs(UKLAD).plot(
    ax=ax,
    color="crimson",
    markersize=8,
    linewidth=0.8,
    label="Automaty paczkowe"
)

ax.set_title("Rozmieszczenie automatów paczkowych w Krakowie")
ax.legend(loc="lower left")
ax.set_axis_off()

fig.savefig(
    FOLDER / "00_start_mapa.png",
    dpi=150,
    bbox_inches="tight"
)

plt.close(fig)

mapa.save(str(FOLDER / "00_start_mapa.html"))

mapa


## 6. Opcjonalnie: ortofotomapa

Dla tematów z obrazami: wycinek ortofotomapy z Geoportalu (usługa WMS GUGiK) w środku Waszych danych. Jeśli obraz się pobierze, dostęp działa; do etykietowania i większych wycinków użyjcie QGIS.

In [8]:
POKAZ_ORTO = False                      # USTAW: True dla tematów z obrazami
WMS_ORTO = "https://mapy.geoportal.gov.pl/wss/service/PZGIK/ORTO/WMS/StandardResolution"
POL_BOKU = 150                          # m: połowa boku wycinka

if POKAZ_ORTO:
    x0, y0, x1, y1 = probka.to_crs(UKLAD).total_bounds   # środek zasięgu danych
    srodek = box(x0, y0, x1, y1).centroid
    parametry = {"SERVICE": "WMS", "VERSION": "1.1.1", "REQUEST": "GetMap", "LAYERS": "Raster", "STYLES": "",
                 "SRS": "EPSG:2180", "FORMAT": "image/jpeg", "WIDTH": 800, "HEIGHT": 800,
                 "BBOX": f"{srodek.x - POL_BOKU},{srodek.y - POL_BOKU},{srodek.x + POL_BOKU},{srodek.y + POL_BOKU}"}
    try:
        odp = requests.get(WMS_ORTO, params=parametry, timeout=60)
        odp.raise_for_status()
        if not odp.headers.get("Content-Type", "").startswith("image"):
            raise ValueError(odp.text[:300])
        (FOLDER / "00_start_orto.jpg").write_bytes(odp.content)
        plt.figure(figsize=(6, 6))
        plt.imshow(plt.imread(io.BytesIO(odp.content), format="jpg"))
        plt.title(f"Ortofotomapa, {2 * POL_BOKU} × {2 * POL_BOKU} m")
        plt.axis("off")
        plt.show()
    except Exception as blad:
        print(f"Usługa WMS nie zwróciła obrazu ({type(blad).__name__}). Sprawdźcie ją w QGIS: {WMS_ORTO}")

## 7. Zapis i co dalej

In [9]:
plik = FOLDER / "00_start.gpkg"
obszar_m.to_file(plik, layer="obszar", driver="GPKG")
zapis = probka.to_crs(UKLAD)
zapis = zapis[[k for k in zapis.columns if k == "geometry" or zapis[k].dtype.kind in "biufO"]]
zapis.astype({k: str for k in zapis.columns if k != "geometry" and zapis[k].dtype.kind == "O"}).to_file(
    plik, layer="probka", driver="GPKG")
(FOLDER / "00_start_metryczka.md").write_text(tabela_metryczki, encoding="utf-8")
print("Zapisano:", plik.name, "(warstwy: obszar, probka),", "00_start_mapa.png,", "00_start_metryczka.md")

Zapisano: 00_start.gpkg (warstwy: obszar, probka), 00_start_mapa.png, 00_start_metryczka.md


In [10]:
kontrola = gpd.read_file(
    FOLDER / "00_start.gpkg",
    layer="probka"
)

print("Układ współrzędnych:", kontrola.crs)
print("Liczba obiektów:", len(kontrola))

Układ współrzędnych: EPSG:2180
Liczba obiektów: 1525


**Co dalej**

- [ ] Metryczkę z sekcji 4 wklejcie do `docs/karta_projektu.md`, a mapę `00_start_mapa.png` do folderu `wyniki`.
- [ ] Jeśli w obszarze projektu leży mniej niż połowa rekordów albo braków jest dużo, opiszcie to w ryzykach karty projektu.
- [ ] Na P2 sprawdźcie licencję danych i wpiszcie ją do karty projektu.
- [ ] Warstwę `00_start.gpkg` możecie otworzyć w QGIS.